# QuantStream Research Lab — model comparison + hyperparameter tuning

**Chạy notebook này trước.** Tạo Dataset version mới từ `QuantStream_Kaggle_Input.zip` mới nhất, chỉ Add Input version đó. Internet On; GPU khả dụng cho TCN/GRU. Không cài lại PyTorch của Kaggle.

Mục tiêu: học cách đặt giả thuyết, so baseline, dùng Optuna và kiểm tra độ ổn định. Chưa có kết quả chạy thật cho notebook này. Không tự chọn champion giao dịch.

- Ridge / XGBoost: baseline tabular; TCN / GRU: mạng chuỗi thời gian.
- 12 trial **tổng cộng**, tối đa 8 epoch cho mỗi lượt train neural. Bốn trial đầu bao phủ đủ bốn họ model; không phải 12 model khác nhau.
- Hai fold sớm dùng xếp hạng. Mỗi neural fold có inner train/early-stop riêng, sau đó refit outer train với epoch cố định.
- Top 2 cấu hình kiểm tra seed 42, 123, 2026 trên hai fold sớm. Ridge là deterministic nên đổi seed không tạo thêm bằng chứng độc lập.
- Khóa lựa chọn trước khi xem fold 3 của candidate. Fold 3 đã từng được xem trong dự án; chỉ là diagnostic development, **không phải holdout độc lập mới**.
- Final holdout cũ đã mở, không xuất trong input và không được gọi ở đây.

In [ ]:
import subprocess,sys,os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'pyarrow>=17,<27','scikit-learn>=1.5,<2','xgboost-cpu>=2.1,<4','optuna>=4.5,<6'])

In [ ]:
from pathlib import Path
import sys, zipfile, shutil, json, hashlib
WORK=Path('/kaggle/working/quantstream_research') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives=list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if len(archives)>1:
    raise ValueError('Chỉ Add Input một phiên bản QuantStream dataset; bỏ các phiên bản cũ.')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts: raise ValueError('Invalid ZIP path')
        z.extractall(WORK)
else:
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1: raise FileNotFoundError('Add Input: QuantStream_Kaggle_Input.zip bản mới nhất.')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
if manifest.get('contains_final_holdout') is not False: raise ValueError('Development-only input required')
if not manifest.get('code_files') or not (WORK/'code/quantstream/search.py').exists():
    raise ValueError('Input cũ: tải lại ZIP mới dành cho notebook 04/05.')
for name,expected in manifest['code_files'].items():
    if hashlib.sha256((WORK/'code'/name).read_bytes()).hexdigest()!=expected:
        raise ValueError('Source checksum mismatch: '+name)
sys.path.insert(0,str(WORK/'code'))
print('Source:',manifest['source_control'])
print('Development only. Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
import torch,pandas as pd
from IPython.display import display,FileLink
from quantstream.search import run_search,load_snapshot
m,dev,bars,cfg=load_snapshot(SNAPSHOT)
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print('Rows:',len(dev),'Test excluded from:',m['split']['test_start'])
display(pd.DataFrame(m['split']['folds']))

## Đóng băng ngân sách và chạy

Objective là trung bình **MSE / MSE của dự báo lợi suất bằng 0** trên hai fold. Nhỏ hơn 1 nghĩa là tốt hơn baseline 0 theo sai số, chưa có nghĩa có lãi. Threshold 0.006 cố định; phí 10 bps và slippage 5 bps mỗi chiều. Không chọn tham số bằng PnL fold 3.

`SEARCH_TIMEOUT=7200` chỉ giới hạn thời gian **bắt đầu trial tìm kiếm**. Trial đang chạy, kiểm tra nhiều seed và diagnostic có thể tiếp tục sau đó; đây không phải cam kết xong trong hai giờ. Nếu thiếu quota, chạy 4 trial / 4 epoch ngay từ đầu trong thí nghiệm riêng và ghi rõ thay đổi.

Giữ nguyên thư mục output để resume. SQLite và CSV giữ cả trial tốt, xấu và bị dừng. Sau khi diagnostic bắt đầu, chương trình từ chối âm thầm chạy lại khi lỗi. Save Version / tải output để tránh mất lịch sử khi Kaggle kết thúc phiên.

In [ ]:
N_TRIALS=12
MAX_EPOCHS=8
SEARCH_TIMEOUT=7200
OUTPUT=WORK/'run-002-model-search'
try:
    result=run_search(SNAPSHOT,OUTPUT,n_trials=N_TRIALS,max_epochs=MAX_EPOCHS,
                      timeout_seconds=SEARCH_TIMEOUT,device='auto')
finally:
    # Preserve partial evidence even if a fit fails or the budget expires.
    if OUTPUT.exists():
        archive=shutil.make_archive(str(WORK.parent/'QuantStream_Research_Output'),'zip',OUTPUT)
        print('Evidence ZIP:',archive)
        display(FileLink(archive))

## Đọc kết quả như researcher

Đọc toàn bộ trial, không chỉ dòng đẹp nhất. So objective giữa seed, số giao dịch, drawdown và ảnh hưởng khi tăng chi phí 1.5x / 2x. Chi phí nhạy cảm hoặc ít giao dịch là kết quả cần ghi nhận, không xóa trial để chạy lại.

Nếu best model thua dự báo 0 hoặc chiến lược âm, kết luận đúng là chưa tìm được tín hiệu hữu ích. Chọn thêm model sau khi xem diagnostic tạo một vòng nghiên cứu mới, không biến fold cũ thành test mới. Không dùng checkpoint TCN đã train toàn development để warm-start các fold lịch sử.

In [ ]:
trials=pd.read_csv(OUTPUT/'trials.csv')
display(trials.sort_values('value').head(20))
display(pd.DataFrame(result['cost_sensitivity']))
print('Selected params:',result['selection']['params'])
print('Across-seed mean/std normalized MSE:',result['selection']['mean_objective'],result['selection']['std_objective'])
print('Status:',result['deployment_status'])
print('Download QuantStream_Research_Output.zip rồi gửi lại để phân tích và commit kết quả thật.')